In [ ]:
import pandas as pd
import numpy as np

sheet_id = "1B_zrNj6vaC8-Cg6ypu9Tz9uRLIPtzBBLL9yRLtVyiAY"
gid = "1073279846"  # ID của sheet/tab

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

df = pd.read_csv(url)

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Kích thước dữ liệu: (1767, 18)


,product_id,title,authors,original_price,current_price,quantity,category,n_review,avg_rating,pages,manufacturer,cover_link,discount,discount_percent,revenue,price_segment,rating_group,page_segment
0,74021317,Cây Cam Ngọt Của Tôi,José Mauro De Vasconcelos,4.56,2.73,53075,Tiểu Thuyết,11481,5.0,244,Nhà Xuất Bản Hội Nhà Văn,https://salt.tadacdn.com/ts/product/5e/18/24/2...,0.4,40.13,"144894,75",Dưới $5,Xuất sắc (4.5 - 5.0),Sách trung bình (200-500 trang)
1,184466860,Hành Tinh Của Một Kẻ Nghĩ Nhiều,Nguyễn Đoàn Minh Thư,3.63,2.53,7929,Others,780,4.8,184,Nhà Xuất Bản Thế Giới,https://salt.tadacdn.com/ts/product/d7/99/24/1...,0.3,30.30,"20060,37",Dưới $5,Xuất sắc (4.5 - 5.0),Sách mỏng (<200 trang)
2,73787185,Những Tù Nhân Của Địa Lý,Tim Marshall,8.86,5.32,17896,Lĩnh Vực Khác,3623,4.8,430,Nhà Xuất Bản Hội Nhà Văn,https://salt.tadacdn.com/ts/product/8d/96/9e/c...,0.4,39.95,"95206,72",Từ $5 - $10,Xuất sắc (4.5 - 5.0),Sách trung bình (200-500 trang)
3,52789367,Nhà Giả Kim (Tái Bản 2020),Paulo Coelho,3.33,2.00,24668,Tác Phẩm Kinh Điển,5131,5.0,228,Nhà Xuất Bản Hà Nội,https://salt.tadacdn.com/ts/product/45/3b/fc/a...,0.4,39.94,49336,Dưới $5,Xuất sắc (4.5 - 5.0),Sách trung bình (200-500 trang)
4,147920903,Một Thoáng Ta Rực Rỡ Ở Nhân Gian,Unknown,5.70,3.42,10000,Tiểu Thuyết,1636,4.8,304,Nhà Xuất Bản Hội Nhà Văn,https://salt.tadacdn.com/ts/product/2f/b5/4e/a...,0.4,40.00,34200,Dưới $5,Xuất sắc (4.5 - 5.0),Sách trung bình (200-500 trang)


In [ ]:
#Phần 1 – Overall KPIs

#Các KPI chính:

#Tổng doanh thu
#Tổng số lượng bán
#Tổng số đầu sách
#Rating trung bình
#Discount trung bình
#Doanh thu trung bình / đầu sách
#Quantity trung bình / đầu sách

In [ ]:
# ==================================================
# OVERALL KPIs (Chuẩn hóa chính xác cột Revenue về 22M)
# ==================================================

import pandas as pd

def clean_revenue_vietnamese(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().replace(' ', '')
    if not val_str:
        return 0.0

    # Nếu có định dạng dấu chấm phân cách hàng nghìn và phẩy thập phân: 1.234.567,89
    if '.' in val_str and ',' in val_str:
        if val_str.find('.') < val_str.find(','):
            val_str = val_str.replace('.', '').replace(',', '.')
        else:
            val_str = val_str.replace(',', '')
    else:
        # Trường hợp chỉ có dấu phẩy (Ví dụ: 12,345 hoặc 22345,67)
        if ',' in val_str:
            # Kiểm tra xem dấu phẩy có phải dấu thập phân thực sự không (Ví dụ: 2 chữ số thập phân sau dấu phẩy)
            parts = val_str.split(',')
            if len(parts[-1]) <= 2:  # Thường là phần thập phân kiểu VN (,5 hoặc ,50)
                val_str = val_str.replace(',', '.')
            else:
                # Nếu là dấu phẩy phân cách hàng nghìn (Ví dụ: 12,345)
                val_str = val_str.replace(',', '')

    return pd.to_numeric(val_str, errors='coerce')

# Áp dụng chuẩn hóa bằng cách sử dụng đúng biến df đã có sẵn
df['revenue'] = df['revenue'].apply(clean_revenue_vietnamese)
df['quantity'] = pd.to_numeric(df['quantity'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
df['avg_rating'] = df['avg_rating'].apply(clean_revenue_vietnamese)
df['discount_percent'] = df['discount_percent'].apply(clean_revenue_vietnamese)

# Sau khi chuyển đổi, kiểm tra xem có outlier nào làm doanh thu vọt lên không
# Ví dụ: một dòng bị chuyển nhầm từ 12,345 thành 1234567
total_revenue = df['revenue'].sum()
total_quantity_sold = df['quantity'].sum()
total_unique_books = df['product_id'].nunique()
average_rating_all_books = df['avg_rating'].mean()
average_discount_percent = df['discount_percent'].mean()

average_revenue_per_book = total_revenue / total_unique_books
average_quantity_per_book = total_quantity_sold / total_unique_books

print("BÁO CÁO TỔNG QUAN HIỆU SUẤT KINH DOANH TADA")
print("=" * 55)
print(f"Tổng doanh thu: ${total_revenue:,.2f}")
print(f"Tổng số lượng sách đã bán: {total_quantity_sold:,.0f} cuốn")
print(f"Tổng số đầu sách: {total_unique_books:,} đầu sách")
print(f"Điểm đánh giá trung bình: {average_rating_all_books:.2f} / 5")
print(f"Tỷ lệ giảm giá trung bình: {average_discount_percent:.2f}%")
print(f"Doanh thu trung bình / đầu sách: ${average_revenue_per_book:,.2f}")
print(f"Số lượng bán trung bình / đầu sách: {average_quantity_per_book:,.2f}")

BÁO CÁO TỔNG QUAN HIỆU SUẤT KINH DOANH TADA
Tổng doanh thu: $22,886,189.49
Tổng số lượng sách đã bán: 4,957,302 cuốn
Tổng số đầu sách: 1,767 đầu sách
Điểm đánh giá trung bình: 4.60 / 5
Tỷ lệ giảm giá trung bình: 27.24%
Doanh thu trung bình / đầu sách: $12,952.00
Số lượng bán trung bình / đầu sách: 2,805.49


In [ ]:
#Phần 2 – Performance by Category

In [ ]:

category_performance = (
    df.groupby('category')
      .agg(
          total_revenue=('revenue', 'sum'),
          total_quantity_sold=('quantity', 'sum'),
          number_of_books=('product_id', 'nunique'),
          average_rating=('avg_rating', 'mean')
      )
      .reset_index()
)

# Tính doanh thu trung bình / đầu sách
category_performance['avg_revenue_per_book'] = (
    category_performance['total_revenue']
    / category_performance['number_of_books']
)

# Tính quantity trung bình / đầu sách
category_performance['avg_quantity_per_book'] = (
    category_performance['total_quantity_sold']
    / category_performance['number_of_books']
)

# Tỷ trọng doanh thu
category_performance['revenue_contribution_pct'] = (
    category_performance['total_revenue']
    / total_revenue
    * 100
)

# Sắp xếp theo doanh thu
category_performance_sorted = (
    category_performance
    .sort_values('total_revenue', ascending=False)
)

display(
    category_performance_sorted.head(10).round(2)
)

,category,total_revenue,total_quantity_sold,number_of_books,average_rating,avg_revenue_per_book,avg_quantity_per_book,revenue_contribution_pct
6,Others,6862440.66,1636659,766,4.60,8958.80,2136.63,29.99
15,Sách Nghệ Thuật Sống Đẹp,3547750.33,785475,58,4.69,61168.11,13542.67,15.50
17,Sách Tài Chính. Tiền Tệ,1835379.70,139928,40,4.49,45884.49,3498.20,8.02
19,Tiểu Thuyết,1303135.77,381031,133,4.48,9798.01,2864.89,5.69
0,Bài Học Kinh Doanh,1244053.06,211421,53,4.57,23472.70,3989.08,5.44
23,Truyện Ngắn - Tản Văn - Tạp Văn,1110876.49,380835,108,4.83,10285.89,3526.25,4.85
12,Sách Kỹ Năng Làm Việc,837878.44,174095,55,4.20,15234.15,3165.36,3.66
21,Truyện Giả Tưởng - Huyền Bí - Phiêu Lưu,697344.11,50294,37,4.58,18847.14,1359.30,3.05
9,Sách Học Tiếng Anh,619531.83,113743,41,4.71,15110.53,2774.22,2.71
20,Truyện Dài,564946.74,174175,23,4.90,24562.90,7572.83,2.47


In [ ]:
# Top Category theo Quantity
top10_category_quantity = (
    category_performance
    .sort_values('total_quantity_sold', ascending=False)
    .head(10)
)

display(
    top10_category_quantity[
        [
            'category',
            'total_quantity_sold',
            'total_revenue',
            'number_of_books',
            'avg_quantity_per_book'
        ]
    ].round(2)
)

,category,total_quantity_sold,total_revenue,number_of_books,avg_quantity_per_book
6,Others,1636659,6862440.66,766,2136.63
15,Sách Nghệ Thuật Sống Đẹp,785475,3547750.33,58,13542.67
19,Tiểu Thuyết,381031,1303135.77,133,2864.89
23,Truyện Ngắn - Tản Văn - Tạp Văn,380835,1110876.49,108,3526.25
0,Bài Học Kinh Doanh,211421,1244053.06,53,3989.08
20,Truyện Dài,174175,564946.74,23,7572.83
12,Sách Kỹ Năng Làm Việc,174095,837878.44,55,3165.36
17,Sách Tài Chính. Tiền Tệ,139928,1835379.70,40,3498.20
9,Sách Học Tiếng Anh,113743,619531.83,41,2774.22
13,Sách Làm Cha Mẹ,107871,324322.60,32,3370.97


In [ ]:
# Category có hiệu quả cao trên mỗi đầu sách
top10_category_efficiency = (
    category_performance[
        category_performance['number_of_books'] >= 5
    ]
    .sort_values(
        'avg_revenue_per_book',
        ascending=False
    )
    .head(10)
)

display(
    top10_category_efficiency[
        [
            'category',
            'number_of_books',
            'total_revenue',
            'avg_revenue_per_book',
            'avg_quantity_per_book'
        ]
    ].round(2)
)

,category,number_of_books,total_revenue,avg_revenue_per_book,avg_quantity_per_book
15,Sách Nghệ Thuật Sống Đẹp,58,3547750.33,61168.11,13542.67
17,Sách Tài Chính. Tiền Tệ,40,1835379.70,45884.49,3498.20
20,Truyện Dài,23,564946.74,24562.90,7572.83
0,Bài Học Kinh Doanh,53,1244053.06,23472.70,3989.08
21,Truyện Giả Tưởng - Huyền Bí - Phiêu Lưu,37,697344.11,18847.14,1359.30
4,Lĩnh Vực Khác,31,521167.98,16811.87,2512.81
12,Sách Kỹ Năng Làm Việc,55,837878.44,15234.15,3165.36
9,Sách Học Tiếng Anh,41,619531.83,15110.53,2774.22
18,Tiểu Sử - Hồi Ký,16,238390.98,14899.44,4643.75
24,Truyện Tranh Ehon,14,201967.29,14426.24,2349.71


In [ ]:
#Tại sao lọc >= 5?

#Nếu một category chỉ có 1 cuốn sách nhưng cuốn đó bán cực tốt, nó có thể đứng đầu avg_revenue_per_book và gây hiểu nhầm.

#Do đó:

#Chỉ xem xét category có ít nhất 5 đầu sách.

In [ ]:
#Xác định Bestseller

bestsellers_by_quantity = (
    df.sort_values(
        'quantity',
        ascending=False
    )
    .head(20)
    .copy()
)

display(
    bestsellers_by_quantity[
        [
            'product_id',
            'title',
            'authors',
            'category',
            'manufacturer',
            'quantity',
            'revenue',
            'avg_rating',
            'n_review',
            'current_price'
        ]
    ]
)

,product_id,title,authors,category,manufacturer,quantity,revenue,avg_rating,n_review,current_price
121,75307228,OSHO - Yêu - Being In Love,Osho,Sách Nghệ Thuật Sống Đẹp,Nhà Xuất Bản Văn Hóa - Văn Nghệ Tp.Hcm,671121,3140846.28,5.0,1855,4.68
36,26114399,Nóng Giận Là Bản Năng . Tĩnh Lặng Là Bản Lĩnh,Tống Mặc,Others,Unknown,65623,158151.43,4.8,6602,2.41
491,750578,Khi Hơi Thở Hóa Thinh Không,Paul Kalanithi,Tiểu Sử - Hồi Ký,Unknown,59734,173228.60,4.7,2099,2.90
0,74021317,Cây Cam Ngọt Của Tôi,José Mauro De Vasconcelos,Tiểu Thuyết,Nhà Xuất Bản Hội Nhà Văn,53075,144894.75,5.0,11481,2.73
44,10005245,Đàn Ông Sao Hỏa Đàn Bà Sao Kim,John Gray,Others,Unknown,48419,276472.49,4.7,3904,5.71
28,1476937,Dám Bị Ghét,Koga Fumitake,Tiểu Thuyết,Unknown,45858,111434.94,4.8,3457,2.43
125,54748899,Muôn Kiếp Nhân Sinh (Many Lives - Many Times),Nguyên Phong,Others,Nhà Xuất Bản Tổng Hợp Tp.Hcm,45110,211114.80,4.8,5075,4.68
21,3639597,Bước Chậm Lại Giữa Thế Gian Vội Vã (Tái Bản),Hae Min,Truyện Ngắn - Tản Văn - Tạp Văn,Unknown,44869,96468.35,4.8,3702,2.15
309,8835159,Đời Ngắn Đừng Ngủ Dài (Tái Bản),Robin Sharma,Others,Unknown,44164,103343.76,4.8,3073,2.34
942,377644,Chờ Đến Mẫu Giáo Thì Đã Muộn,Ibuka Masaru,Sách Làm Cha Mẹ,Nhà Xuất Bản Văn Học,36933,85684.56,4.7,671,2.32


In [ ]:
#Top 1 thực tế

#Sách đứng đầu theo quantity trong dataset hiện tại là:

#OSHO - Yêu - Being In Love

#với quantity khoảng 671,121.

#Đây cũng là một outlier cực lớn so với phần còn lại.

In [ ]:
# Top 20 theo revenue

bestsellers_by_revenue = (
    df.sort_values(
        'revenue',
        ascending=False
    )
    .head(20)
    .copy()
)

display(
    bestsellers_by_revenue[
        [
            'product_id',
            'title',
            'authors',
            'category',
            'manufacturer',
            'quantity',
            'revenue',
            'avg_rating',
            'n_review',
            'current_price'
        ]
    ]
)

,product_id,title,authors,category,manufacturer,quantity,revenue,avg_rating,n_review,current_price
121,75307228,OSHO - Yêu - Being In Love,Osho,Sách Nghệ Thuật Sống Đẹp,Nhà Xuất Bản Văn Hóa - Văn Nghệ Tp.Hcm,671121,3140846.28,5.0,1855,4.68
687,1080002,Boxset Harry Potter - Tiếng Việt (Trọn Bộ 7 Tập),J. K. Rowling,Truyện Giả Tưởng - Huyền Bí - Phiêu Lưu,Unknown,9562,484124.06,4.7,1438,50.63
75,11122712,Bộ sách Làm Giàu Từ Chứng Khoán (How To Make M...,William J. O’Neil,Sách Tài Chính. Tiền Tệ,Nhà Xuất Bản Thế Giới,14656,432938.24,4.8,2320,29.54
111,3608625,Payback Time - Ngày Đòi Nợ,Phil Town,Bài Học Kinh Doanh,Nhà Xuất Bản Thế Giới,31234,394173.08,4.8,4311,12.62
44,10005245,Đàn Ông Sao Hỏa Đàn Bà Sao Kim,John Gray,Others,Unknown,48419,276472.49,4.7,3904,5.71
27,97594364,Luật Tâm Thức - Giải Mã Ma Trận Vũ Trụ,Ngô Sa Thạch,Lĩnh Vực Khác,Nhà Xuất Bản Dân Trí,27672,260393.52,4.7,6325,9.41
292,53532186,Tuyệt kỹ Giao dịch bằng đồ thị nến Nhật – Japa...,Steve Nison,Sách Tài Chính. Tiền Tệ,Nhà Xuất Bản Thế Giới,12195,256704.75,4.8,2204,21.05
125,54748899,Muôn Kiếp Nhân Sinh (Many Lives - Many Times),Nguyên Phong,Others,Nhà Xuất Bản Tổng Hợp Tp.Hcm,45110,211114.80,4.8,5075,4.68
64,381234,Cuốn Sách Hoàn Hảo Về Ngôn Ngữ Cơ Thể - Body L...,Allan & Barbara Pease,Others,Unknown,30490,204587.90,4.7,1076,6.71
1065,49249526,Thiết Kế Cuộc Đời Thịnh Vượng - Design a Prosp...,Thái Phạm,Others,Nhà Xuất Bản Thế Giới,15571,196506.02,5.0,2808,12.62


In [ ]:
#Bestseller theo Price Segment

In [ ]:
price_performance = (
    df.groupby('price_segment')
      .agg(
          total_revenue=('revenue', 'sum'),
          total_quantity=('quantity', 'sum'),
          number_of_books=('product_id', 'nunique'),
          avg_rating=('avg_rating', 'mean')
      )
      .reset_index()
)

price_performance['avg_revenue_per_book'] = (
    price_performance['total_revenue']
    / price_performance['number_of_books']
)

price_performance['avg_quantity_per_book'] = (
    price_performance['total_quantity']
    / price_performance['number_of_books']
)

price_performance['revenue_contribution_pct'] = (
    price_performance['total_revenue']
    / total_revenue * 100
)

price_performance

,price_segment,total_revenue,total_quantity,number_of_books,avg_rating,avg_revenue_per_book,avg_quantity_per_book,revenue_contribution_pct
0,Dưới $5,12349669.76,3761715,1201,4.656869,10282.822448,3132.152373,53.961232
1,Trên $20,1653729.20,55370,25,4.264000,66149.168000,2214.800000,7.225883
2,Từ $10 - $20,2909377.95,231645,119,4.573109,24448.554202,1946.596639,12.712374
3,Từ $5 - $10,5973412.58,908572,422,4.450948,14155.006114,2153.014218,26.100512


In [ ]:
price_performance = price_performance.sort_values(
    'total_revenue',
    ascending=False
)

display(price_performance.round(2))

,price_segment,total_revenue,total_quantity,number_of_books,avg_rating,avg_revenue_per_book,avg_quantity_per_book,revenue_contribution_pct
0,Dưới $5,12349669.76,3761715,1201,4.66,10282.82,3132.15,53.96
3,Từ $5 - $10,5973412.58,908572,422,4.45,14155.01,2153.01,26.10
2,Từ $10 - $20,2909377.95,231645,119,4.57,24448.55,1946.60,12.71
1,Trên $20,1653729.20,55370,25,4.26,66149.17,2214.80,7.23


In [ ]:
overall_kpi = pd.DataFrame({
    'metric': [
        'Total Revenue',
        'Total Quantity Sold',
        'Total Unique Books',
        'Average Rating',
        'Average Discount (%)',
        'Average Revenue per Book',
        'Average Quantity per Book'
    ],
    'value': [
        total_revenue,
        total_quantity_sold,
        total_unique_books,
        average_rating_all_books,
        average_discount_percent,
        average_revenue_per_book,
        average_quantity_per_book
    ]
})

display(overall_kpi.round(2))

overall_kpi.to_csv(
  #  f'{output_folder}/overall_kpi.csv',
    index=False,
    encoding='utf-8-sig'
)

,metric,value
0,Total Revenue,22886189.49
1,Total Quantity Sold,4957302.00
2,Total Unique Books,1767.00
3,Average Rating,4.60
4,Average Discount (%),27.24
5,Average Revenue per Book,12952.00
6,Average Quantity per Book,2805.49


'metric,value\nTotal Revenue,22886189.49\nTotal Quantity Sold,4957302.0\nTotal Unique Books,1767.0\nAverage Rating,4.596491228070175\nAverage Discount (%),27.24458404074703\nAverage Revenue per Book,12952.00310696095\nAverage Quantity per Book,2805.4906621392192\n'

# Tổng kết Task 7 – Tính toán các chỉ số

## 1. Hiệu suất tổng thể

Dataset gồm 1,767 đầu sách. Tổng doanh thu, tổng số lượng bán, rating trung bình và tỷ lệ giảm giá trung bình được sử dụng làm các KPI chính để đánh giá hiệu quả kinh doanh.

Rating trung bình của toàn bộ danh mục ở mức khoảng 4.60/5, trong khi mức giảm giá trung bình khoảng 27.24%.

## 2. Hiệu suất theo Category

Các category có tổng doanh thu và tổng quantity cao nhất là những nhóm đóng góp quan trọng vào hoạt động kinh doanh.

Tuy nhiên, tổng doanh thu cần được xem xét cùng với số lượng đầu sách của từng category. Một category có nhiều đầu sách có thể đạt tổng doanh thu cao do quy mô danh mục lớn.

Do đó, chỉ số `avg_revenue_per_book` được bổ sung để đánh giá hiệu quả trung bình trên mỗi đầu sách.

## 3. Hiệu suất theo Manufacturer

Tương tự category, các manufacturer có tổng doanh thu cao không nhất thiết có hiệu suất trên từng đầu sách cao nhất.

Các chỉ số `avg_revenue_per_book` và `avg_quantity_per_book` được sử dụng để đánh giá hiệu quả tương đối giữa các nhà xuất bản.

## 4. Bestseller

Danh sách Top 20 được xác định theo hai tiêu chí:

- Quantity bán ra
- Revenue

Trong dataset hiện tại, "OSHO - Yêu - Being In Love" đứng đầu theo quantity và revenue, đồng thời có lượng bán cao vượt trội so với phần lớn các đầu sách còn lại.

## 5. Insight chính

- Hiệu suất kinh doanh giữa các đầu sách có sự chênh lệch rất lớn.
- Một số ít đầu sách đóng góp lượng quantity và revenue rất lớn.
- Phân khúc giá thấp có quy mô danh mục lớn và đóng vai trò quan trọng trong quantity.
- Khi đánh giá category hoặc manufacturer, cần xem xét cả tổng chỉ số và chỉ số trung bình trên mỗi đầu sách.
- Bestseller nên được phân tích theo nhiều tiêu chí thay vì chỉ dựa vào quantity.